In [1]:
!pip install -q pixeltable-pgserver psycopg2-binary sqlalchemy pandas gdown

import os
import warnings
import urllib.request

import pandas as pd
from sqlalchemy import create_engine, text

try:
    import pgserver
except ImportError:
    import pixeltable_pgserver as pgserver

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)

pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
engine = create_engine(
    pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://'),
    isolation_level='AUTOCOMMIT',
    future=True,
)
conn = engine.connect()
print(conn.execute(text('SELECT version()')).scalar_one())

def sql(query):
    return pd.read_sql(text(query), conn)

def run(query):
    conn.exec_driver_sql(query)

PostgreSQL 16.15 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


## 1. Завантаження даних

In [2]:
CSV_PATH = 'data/infectious_cases.csv'
GDRIVE_ID = '1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl'
RAW_URL = 'https://raw.githubusercontent.com/qEmrys/goit-rdb-fp/main/data/infectious_cases.csv'

def csv_ok(path):
    if not os.path.exists(path):
        return False
    with open(path, encoding='utf-8') as f:
        return f.readline().startswith('Entity,Code,Year')

if not csv_ok(CSV_PATH):
    os.makedirs('data', exist_ok=True)
    try:
        import gdown
        gdown.download(id=GDRIVE_ID, output=CSV_PATH, quiet=True)
    except Exception as e:
        print('gdown не спрацював:', type(e).__name__)
if not csv_ok(CSV_PATH):
    urllib.request.urlretrieve(RAW_URL, CSV_PATH)

print(CSV_PATH, os.path.getsize(CSV_PATH), 'байт')

data/infectious_cases.csv 540895 байт


In [3]:
run('''
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
''')
print('search_path:', conn.execute(text('SHOW search_path')).scalar_one())
print('current_schema:', conn.execute(text('SELECT current_schema()')).scalar_one())

search_path: pandemic
current_schema: pandemic


In [4]:
run('''
DROP TABLE IF EXISTS infectious_cases CASCADE;

CREATE TABLE infectious_cases (
    entity TEXT,
    code TEXT,
    year INTEGER,
    number_yaws TEXT,
    polio_cases TEXT,
    cases_guinea_worm TEXT,
    number_rabies TEXT,
    number_malaria TEXT,
    number_hiv TEXT,
    number_tuberculosis TEXT,
    number_smallpox TEXT,
    number_cholera_cases TEXT
);
''')

with conn.connection.cursor() as cur, open(CSV_PATH, encoding='utf-8') as f:
    cur.copy_expert('COPY infectious_cases FROM STDIN WITH (FORMAT csv, HEADER true)', f)
print('Імпортовано рядків:', conn.execute(text('SELECT COUNT(*) FROM infectious_cases')).scalar_one())

Імпортовано рядків: 10521


In [5]:
sql(r"""
SELECT
    COUNT(*)::BIGINT AS n_rows,
    COUNT(DISTINCT entity)::BIGINT AS n_entity,
    COUNT(DISTINCT code)::BIGINT AS n_code,
    MIN(year) AS min_year,
    MAX(year) AS max_year
FROM infectious_cases;
""")

,n_rows,n_entity,n_code,min_year,max_year
0,10521,245,211,1920,2023


In [6]:
sql(r"""
SELECT *
FROM infectious_cases
WHERE number_rabies IS NOT NULL
LIMIT 10;
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1990,None,336,0,2.145772,1219845.8,87.665596,16303.112,None,NaN
1,Afghanistan,AFG,1991,None,14,0,2.3209748,1132781.5,103.77359,17582.254,None,NaN
2,Afghanistan,AFG,1995,None,0,0,3.126361,725758.3,143.54527,22163.428,None,19903
3,Afghanistan,AFG,1997,None,21,0,3.446122,605853.06,160.59525,23140.863,None,4170
4,Afghanistan,AFG,1998,None,65,0,3.5570247,547045.4,164.60725,23179.447,None,10000
5,Afghanistan,AFG,1999,None,167,0,3.607037,502898.88,168.99359,22944.182,None,24639
6,Afghanistan,AFG,2000,None,133,0,3.6362348,491614.72,177.66675,22879.678,None,4330
7,Afghanistan,AFG,2001,None,22,0,3.7118323,511626.2,192.51457,23330.46,None,4499
8,Afghanistan,AFG,2002,None,11,0,3.841707,546030.6,208.17296,24148.05,None,3
9,Afghanistan,AFG,2003,None,9,0,3.914814,574740.3,220.20476,24519.38,None,41


In [7]:
sql(r"""
SELECT
    c.column_name,
    c.data_type,
    COUNT(e.v)::BIGINT AS n_not_null,
    (COUNT(*) - COUNT(e.v))::BIGINT AS n_null
FROM infectious_cases AS t
CROSS JOIN LATERAL jsonb_each_text(to_jsonb(t)) AS e(k, v)
JOIN information_schema.columns AS c
    ON c.table_schema = 'pandemic'
   AND c.table_name = 'infectious_cases'
   AND c.column_name = e.k
GROUP BY c.column_name, c.data_type, c.ordinal_position
ORDER BY c.ordinal_position;
""")

,column_name,data_type,n_not_null,n_null
0,entity,text,10521,0
1,code,text,9393,1128
2,year,integer,10521,0
3,number_yaws,text,74,10447
4,polio_cases,text,7610,2911
5,cases_guinea_worm,text,8428,2093
6,number_rabies,text,6780,3741
7,number_malaria,text,6780,3741
8,number_hiv,text,6780,3741
9,number_tuberculosis,text,6780,3741


Імпортовано 10521 рядок, 12 колонок, 245 Entity, роки 1920–2023. Number_* завантажив як TEXT, бо в CSV багато пустих значень: number_rabies заповнений у 6780 рядках, пустий у 3741. Code пустий у 1128 рядках, це агрегати на зразок Africa або African Region (WHO).

## 2. Нормалізація до 3НФ

In [8]:
run(r"""
CREATE OR REPLACE FUNCTION text_to_numeric(p_value TEXT)
RETURNS NUMERIC
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT CASE
        WHEN btrim(p_value) ~ '^-?[0-9]+(\.[0-9]+)?([eE][-+]?[0-9]+)?$'
            THEN btrim(p_value)::NUMERIC
    END;
$$;

DROP TABLE IF EXISTS infectious_cases_normalized CASCADE;
DROP TABLE IF EXISTS entities CASCADE;

CREATE TABLE entities (
    id SERIAL PRIMARY KEY,
    entity TEXT NOT NULL,
    code TEXT,
    UNIQUE NULLS NOT DISTINCT (entity, code)
);

CREATE TABLE infectious_cases_normalized (
    id SERIAL PRIMARY KEY,
    entity_id INTEGER NOT NULL REFERENCES entities(id),
    year INTEGER NOT NULL,
    number_yaws NUMERIC,
    polio_cases NUMERIC,
    cases_guinea_worm NUMERIC,
    number_rabies NUMERIC,
    number_malaria NUMERIC,
    number_hiv NUMERIC,
    number_tuberculosis NUMERIC,
    number_smallpox NUMERIC,
    number_cholera_cases NUMERIC,
    UNIQUE (entity_id, year)
);
""")
print('Таблиці entities і infectious_cases_normalized створено')

Таблиці entities і infectious_cases_normalized створено


In [9]:
run(r"""
INSERT INTO entities (entity, code)
SELECT DISTINCT entity, NULLIF(btrim(code), '')
FROM infectious_cases
ORDER BY 1, 2;

INSERT INTO infectious_cases_normalized (entity_id, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies, number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases)
SELECT
    e.id,
    ic.year,
    text_to_numeric(ic.number_yaws),
    text_to_numeric(ic.polio_cases),
    text_to_numeric(ic.cases_guinea_worm),
    text_to_numeric(ic.number_rabies),
    text_to_numeric(ic.number_malaria),
    text_to_numeric(ic.number_hiv),
    text_to_numeric(ic.number_tuberculosis),
    text_to_numeric(ic.number_smallpox),
    text_to_numeric(ic.number_cholera_cases)
FROM infectious_cases AS ic
JOIN entities AS e
    ON e.entity = ic.entity
   AND e.code IS NOT DISTINCT FROM NULLIF(btrim(ic.code), '')
ORDER BY e.id, ic.year;
""")
print('Дані перенесено')

Дані перенесено


In [10]:
sql(r"""
SELECT table_name, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name IN ('entities', 'infectious_cases_normalized')
ORDER BY table_name, ordinal_position;
""")

,table_name,column_name,data_type,is_nullable
0,entities,id,integer,NO
1,entities,entity,text,NO
2,entities,code,text,YES
3,infectious_cases_normalized,id,integer,NO
4,infectious_cases_normalized,entity_id,integer,NO
5,infectious_cases_normalized,year,integer,NO
6,infectious_cases_normalized,number_yaws,numeric,YES
7,infectious_cases_normalized,polio_cases,numeric,YES
8,infectious_cases_normalized,cases_guinea_worm,numeric,YES
9,infectious_cases_normalized,number_rabies,numeric,YES


In [11]:
sql(r"""
SELECT *
FROM entities
ORDER BY id
LIMIT 5;
""")

,id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,NaN
2,3,African Region (WHO),NaN
3,4,Albania,ALB
4,5,Algeria,DZA


In [12]:
sql(r"""
SELECT *
FROM infectious_cases_normalized
WHERE number_rabies IS NOT NULL
ORDER BY id
LIMIT 5;
""")

,id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,13,1,1990,None,336.0,0.0,2.145772,1219845.8,87.665596,16303.112,None,NaN
1,14,1,1991,None,14.0,0.0,2.320975,1132781.5,103.773590,17582.254,None,NaN
2,15,1,1992,None,NaN,0.0,2.618041,1081609.6,119.933400,19786.672,None,NaN
3,16,1,1993,None,NaN,0.0,2.853117,963817.1,127.263300,21195.934,None,37046.0
4,17,1,1994,None,NaN,0.0,2.989987,818779.5,133.063080,21658.172,None,38735.0


In [13]:
sql(r"""
SELECT
    (SELECT COUNT(*) FROM infectious_cases)::BIGINT AS n_raw,
    (SELECT COUNT(*) FROM infectious_cases_normalized)::BIGINT AS n_normalized,
    (SELECT COUNT(*) FROM (SELECT DISTINCT entity, code FROM infectious_cases) AS d)::BIGINT AS n_entity_code_raw,
    (SELECT COUNT(*) FROM entities)::BIGINT AS n_entities,
    (SELECT COUNT(*) FROM (
        SELECT entity, COUNT(DISTINCT COALESCE(code, '')) AS n
        FROM infectious_cases
        GROUP BY entity
        HAVING COUNT(DISTINCT COALESCE(code, '')) > 1
    ) AS d)::BIGINT AS n_entity_with_2plus_codes,
    (SELECT COUNT(*)
     FROM infectious_cases AS t
     CROSS JOIN LATERAL jsonb_each_text(to_jsonb(t)) AS e(k, v)
     WHERE e.k NOT IN ('entity', 'code', 'year')
       AND e.v IS NOT NULL
       AND text_to_numeric(e.v) IS NULL)::BIGINT AS n_non_numeric_values;
""")

,n_raw,n_normalized,n_entity_code_raw,n_entities,n_entity_with_2plus_codes,n_non_numeric_values
0,10521,10521,245,245,0,0


In [14]:
sql(r"""
WITH src AS (
    SELECT
        entity,
        NULLIF(btrim(code), '') AS code,
        year,
        text_to_numeric(number_yaws) AS number_yaws,
        text_to_numeric(polio_cases) AS polio_cases,
        text_to_numeric(cases_guinea_worm) AS cases_guinea_worm,
        text_to_numeric(number_rabies) AS number_rabies,
        text_to_numeric(number_malaria) AS number_malaria,
        text_to_numeric(number_hiv) AS number_hiv,
        text_to_numeric(number_tuberculosis) AS number_tuberculosis,
        text_to_numeric(number_smallpox) AS number_smallpox,
        text_to_numeric(number_cholera_cases) AS number_cholera_cases
    FROM infectious_cases
),
back AS (
    SELECT
        e.entity,
        e.code,
        n.year,
        n.number_yaws,
        n.polio_cases,
        n.cases_guinea_worm,
        n.number_rabies,
        n.number_malaria,
        n.number_hiv,
        n.number_tuberculosis,
        n.number_smallpox,
        n.number_cholera_cases
    FROM infectious_cases_normalized AS n
    JOIN entities AS e ON e.id = n.entity_id
)
SELECT
    (SELECT COUNT(*) FROM (SELECT * FROM src EXCEPT ALL SELECT * FROM back) AS a)::BIGINT AS src_except_back,
    (SELECT COUNT(*) FROM (SELECT * FROM back EXCEPT ALL SELECT * FROM src) AS b)::BIGINT AS back_except_src;
""")

,src_except_back,back_except_src
0,0,0


Entity і Code повторюються в кожному рядку року, тому виніс їх в entities (245 рядків). Code у 34 записів NULL, тому UNIQUE NULLS NOT DISTINCT. В infectious_cases_normalized залишились entity_id, year і показники, пара (entity_id, year) унікальна. Кожна Entity має рівно один Code, тож транзитивних залежностей в таблиці показників немає.

Number_* при перенесенні перевів у NUMERIC через text_to_numeric: пусті значення стають NULL, нечислових значень 0. Рядків до і після 10521, EXCEPT ALL в обидва боки дає 0, тобто JOIN назад відновлює початкову таблицю.

## 3. Аналіз Number_rabies

In [15]:
sql(r"""
SELECT
    e.id,
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(n.number_rabies), 2) AS min_rabies,
    ROUND(MAX(n.number_rabies), 2) AS max_rabies,
    ROUND(SUM(n.number_rabies), 2) AS sum_rabies
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10;
""")

,id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,241,World,OWID_WRL,20192.37,14075.51,24744.66,605771.11
1,121,Lower Middle Income (WB),NaN,15193.96,10202.53,19182.80,455818.78
2,202,South Asia (WB),NaN,11729.89,7271.28,15361.88,351896.66
3,206,South-East Asia Region (WHO),NaN,11424.33,6806.01,15641.96,342729.81
4,77,G20,NaN,10189.05,6339.08,13164.88,305671.38
5,97,India,IND,8599.17,5425.87,11121.14,257975.20
6,209,Sub-Saharan Africa (WB),NaN,4748.27,4356.82,5017.98,142448.04
7,3,African Region (WHO),NaN,4649.70,4283.07,4917.78,139491.07
8,120,Low Income (WB),NaN,3568.11,3014.62,3873.53,107043.16
9,58,East Asia & Pacific (WB),NaN,3427.46,2009.94,4591.01,102823.83


У top-10 переважно агрегати: World, групи за доходом, регіони WB і WHO, з країн тільки India. WHERE number_rabies IS NOT NULL потрібен: у 19 Entity немає жодного значення, їх AVG = NULL і при DESC вони стали б першими. Окремо порахував тільки країни (без NULL Code і OWID_*).

In [16]:
sql(r"""
SELECT
    e.id,
    e.entity,
    e.code,
    ROUND(AVG(n.number_rabies), 2) AS avg_rabies,
    ROUND(MIN(n.number_rabies), 2) AS min_rabies,
    ROUND(MAX(n.number_rabies), 2) AS max_rabies,
    ROUND(SUM(n.number_rabies), 2) AS sum_rabies
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
  AND e.code IS NOT NULL
  AND e.code NOT LIKE 'OWID%'
GROUP BY e.id, e.entity, e.code
ORDER BY AVG(n.number_rabies) DESC
LIMIT 10;
""")

,id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,97,India,IND,8599.17,5425.87,11121.14,257975.20
1,163,Pakistan,PAK,1582.17,1177.14,1881.73,47465.09
2,151,Nigeria,NGA,1335.42,1073.95,1441.12,40062.46
3,42,China,CHN,1247.91,744.25,2075.59,37437.20
4,70,Ethiopia,ETH,1145.17,758.05,1323.40,34355.18
5,143,Myanmar,MMR,972.90,560.70,1449.83,29187.11
6,19,Bangladesh,BGD,785.71,180.40,1734.71,23571.26
7,146,Nepal,NPL,661.25,338.14,1041.63,19837.53
8,170,Philippines,PHL,618.69,358.06,898.07,18560.73
9,98,Indonesia,IDN,267.71,124.62,481.37,8031.44


Серед країн перша India (avg 8599.17), далі Pakistan (1582.17) і Nigeria (1335.42).

## 4. Різниця в роках

In [17]:
sql(r"""
SELECT
    n.id,
    e.entity,
    n.year,
    MAKE_DATE(n.year, 1, 1) AS year_start_date,
    CURRENT_DATE AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INTEGER AS years_diff
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;
""")

,id,entity,year,year_start_date,today,years_diff
0,1,Afghanistan,1960,1960-01-01,2026-11-23,66
1,2,Afghanistan,1965,1965-01-01,2026-11-23,61
2,3,Afghanistan,1980,1980-01-01,2026-11-23,46
3,4,Afghanistan,1981,1981-01-01,2026-11-23,45
4,5,Afghanistan,1982,1982-01-01,2026-11-23,44
5,6,Afghanistan,1983,1983-01-01,2026-11-23,43
6,7,Afghanistan,1984,1984-01-01,2026-11-23,42
7,8,Afghanistan,1985,1985-01-01,2026-11-23,41
8,9,Afghanistan,1986,1986-01-01,2026-11-23,40
9,10,Afghanistan,1987,1987-01-01,2026-11-23,39


In [18]:
sql(r"""
SELECT
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    MAX(EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER) AS max_years_diff,
    MIN(EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER) AS min_years_diff
FROM infectious_cases_normalized;
""")

,min_year,max_year,max_years_diff,min_years_diff
0,1920,2023,106,3


Дату 1 січня будую через MAKE_DATE, поточну через CURRENT_DATE, різницю через AGE і EXTRACT(YEAR). Прогон 2026-11-23: для 1960 різниця 66 років, по всій таблиці від 3 (2023) до 106 (1920).

## 5. Власні функції

In [19]:
run(r"""
CREATE OR REPLACE FUNCTION fn_years_diff(p_year INTEGER)
RETURNS INTEGER
LANGUAGE plpgsql
STABLE
AS $$
BEGIN
    IF p_year IS NULL THEN
        RETURN NULL;
    END IF;
    RETURN EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
END;
$$;

CREATE OR REPLACE FUNCTION fn_avg_per_period(p_cases NUMERIC, p_divider NUMERIC)
RETURNS NUMERIC
LANGUAGE plpgsql
IMMUTABLE
AS $$
BEGIN
    IF p_cases IS NULL OR p_divider IS NULL OR p_divider = 0 THEN
        RETURN NULL;
    END IF;
    RETURN p_cases / p_divider;
END;
$$;
""")
print('Функції fn_years_diff і fn_avg_per_period створено')

Функції fn_years_diff і fn_avg_per_period створено


In [20]:
sql(r"""
SELECT
    fn_years_diff(1996) AS years_diff_1996,
    ROUND(fn_avg_per_period(1200, 12), 2) AS per_month_1200,
    ROUND(fn_avg_per_period(1200, 4), 2) AS per_quarter_1200,
    ROUND(fn_avg_per_period(1200, 2), 2) AS per_half_year_1200,
    ROUND(fn_avg_per_period(1200, 0), 2) AS divider_0;
""")

,years_diff_1996,per_month_1200,per_quarter_1200,per_half_year_1200,divider_0
0,30,100.0,300.0,600.0,None


In [21]:
sql(r"""
SELECT
    n.id,
    e.entity,
    n.year,
    fn_years_diff(n.year) AS years_diff
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;
""")

,id,entity,year,years_diff
0,1,Afghanistan,1960,66
1,2,Afghanistan,1965,61
2,3,Afghanistan,1980,46
3,4,Afghanistan,1981,45
4,5,Afghanistan,1982,44
5,6,Afghanistan,1983,43
6,7,Afghanistan,1984,42
7,8,Afghanistan,1985,41
8,9,Afghanistan,1986,40
9,10,Afghanistan,1987,39


In [22]:
sql(r"""
SELECT
    COUNT(*)::BIGINT AS n_rows,
    COUNT(*) FILTER (
        WHERE fn_years_diff(year) <> EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER
    )::BIGINT AS n_mismatch
FROM infectious_cases_normalized;
""")

,n_rows,n_mismatch
0,10521,0


In [23]:
sql(r"""
SELECT
    n.id,
    e.entity,
    n.year,
    ROUND(n.number_rabies, 2) AS rabies_per_year,
    ROUND(fn_avg_per_period(n.number_rabies, 12), 2) AS rabies_per_month,
    ROUND(fn_avg_per_period(n.number_rabies, 4), 2) AS rabies_per_quarter,
    ROUND(fn_avg_per_period(n.number_rabies, 2), 2) AS rabies_per_half_year
FROM infectious_cases_normalized AS n
JOIN entities AS e ON e.id = n.entity_id
WHERE n.number_rabies IS NOT NULL
ORDER BY n.id
LIMIT 10;
""")

,id,entity,year,rabies_per_year,rabies_per_month,rabies_per_quarter,rabies_per_half_year
0,13,Afghanistan,1990,2.15,0.18,0.54,1.07
1,14,Afghanistan,1991,2.32,0.19,0.58,1.16
2,15,Afghanistan,1992,2.62,0.22,0.65,1.31
3,16,Afghanistan,1993,2.85,0.24,0.71,1.43
4,17,Afghanistan,1994,2.99,0.25,0.75,1.49
5,18,Afghanistan,1995,3.13,0.26,0.78,1.56
6,19,Afghanistan,1996,3.28,0.27,0.82,1.64
7,20,Afghanistan,1997,3.45,0.29,0.86,1.72
8,21,Afghanistan,1998,3.56,0.30,0.89,1.78
9,22,Afghanistan,1999,3.61,0.30,0.90,1.80


In [24]:
sql(r"""
SELECT
    COUNT(*)::BIGINT AS n_rows,
    COUNT(*) FILTER (WHERE number_rabies IS NOT NULL)::BIGINT AS n_numeric_rabies,
    COUNT(*) FILTER (WHERE fn_avg_per_period(number_rabies, 12) IS NOT NULL)::BIGINT AS n_with_result
FROM infectious_cases_normalized;
""")

,n_rows,n_numeric_rabies,n_with_result
0,10521,6780,6780


fn_years_diff(1996) повертає 30, по всій таблиці результат збігається з виразом із завдання 4 (0 розбіжностей на 10521 рядку). fn_avg_per_period(1200, 12/4/2) дає 100, 300 і 600, при дільнику 0 повертає NULL замість помилки. Для другої функції брав тільки рядки з числовим number_rabies, їх 6780, для всіх функція повернула значення.